# 07 - Figure 3

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
from src import data, preprocess, config
from src.plotting import *   # shared figure style defaults

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
from matplotlib.patches import Rectangle

# Parameters

In [ ]:
save = True
optimal_cluster = 'st_cluster_final_gid' # 'st_cluster_final_gid' or 'st_cluster_3_5_7'

test_case = 'block'
historic_ref = '' # '_Full' '_Discount' or ''
premium_type = '' # '_Full'  '_Discount' or ''

use_reinsurance = True
base_case = ''
if use_reinsurance == False:
    base_case = '_base_case'

In [ ]:
if test_case == 'historic':
    simulations = 1
else:
    steps = 100
    simulations = 1000

# Data Load

## Simulations

In [ ]:
res = data.load_simulation_results(test_case=test_case, premium_type=premium_type,
                                   base_case=base_case, historic_ref=historic_ref)
balance_transition_df = res['balance_transition']

## Geospatial

In [ ]:
gdf_states = data.load_states()
gdf_rivers = data.load_rivers()
gdf_states["STATEFP"] = pd.to_numeric(gdf_states["STATEFP"], errors="coerce").astype("Int64")

## NFIP Clustered Claims

In [ ]:
clustered_claims, optimal_cluster = data.load_clustered_claims()

In [ ]:
clustered_claims = preprocess.add_claim_fields(clustered_claims)

In [ ]:
_cpi = data.load_cpi_annual()
clustered_claims = preprocess.cpi_adjust_claims(clustered_claims, _cpi)

# Plotting

## Relative Hyperclusters

In [ ]:
# Filter to years after 1987 (10 years)
post_initial_period = balance_transition_df[balance_transition_df["year"] > 1987]

# Count how many simulations each cluster appears in
cluster_sim_counts = (
    post_initial_period
    .groupby("top_cluster")["simulation"]
    .nunique()
)

# Filter to only clusters that appear in at least 20% of simulations
if test_case == "historic":
    frequent_clusters = cluster_sim_counts[cluster_sim_counts >= 1].index
else:
    num_thres = simulations*0.2
    frequent_clusters = cluster_sim_counts[cluster_sim_counts >= num_thres].index

# Convert to set
events = set(frequent_clusters)
events = (events - {1927}) | {1923}

print(f'{len(events)} Imbalancing events:{events}')

In [ ]:
cluster_event_map = config.CLUSTER_EVENT_MAP
covered_events = config.COVERED_NEW

# Identified across simulation experiments - cluster specific
critical_events = [941, 1171, 1734, 1895, 1923, 2112, 2214, 2578]
critical_states = {
    1923: ['48', '17', '29'],
    1676: ['01', '42', '54', '39'],
    2578: ['48'],
    1171: ['46', '27','38'],
    546: ['54', '51'],
    2214: ['44', '36', '34', '9'],
    1835: ['40','20'],
    941: ['29','31','17','19','20'], 
    2486: ['54'],
    2112: ['46','5','29','30','38'],
    1734: ['28', '22', '01'],
    1355: ['11','37','42'],
    1101: ['11','42','54'],
    462: ['40'],
    2138: ['50','42','37','36','34','09'],
    1895: ['18','19','55'],
    2410: ['40'],
    2040: ['47', '21'],
    2687: ['5', '29', '40'],
}

In [ ]:
# normalize critical_states FIPS to 2-digit strings
def _norm_fips_list(lst):
    # Handles strings like '9', '05', ints, etc.
    return [str(int(x)).zfill(2) for x in lst]

normalized_critical_states = {
    k: _norm_fips_list(v) for k, v in critical_states.items()
}

# Styling choices
POINT_COLOR_DEFAULT = '#274B51'
POINT_COLOR_CRITICAL = '#C4403A'   # purple
COVERED_FACE = '#F9DCE0'         # light red background for covered events
STATE_FILL = '#593E42'             # light brown state shading, #564348
RIVER_FILL = '#40909A'             # light blue shading, '#C1EAFD'
STATE_FILL_ALPHA = 0.45

# Convert events to sorted list for consistent ordering
event_list = sorted(events)

# Set spatial extent for consistency
extent = [-102, -65, 24, 50]

# Adjust grid if more events are needed
n_panels = len(event_list)
nrows = 5
ncols = int(np.ceil(n_panels / nrows))

label_offset = (-0.2, 1.1)

# Prepare figure
fig, axs = plt.subplots(nrows, ncols, figsize=(7, 7), constrained_layout=True)
axs = axs.flatten()
panel_labels = [f"{chr(97 + i)})" for i in range(len(axs))]

for i, cluster_id in enumerate(event_list):
    ax = axs[i]

    # background shading for covered events
    if cluster_id in covered_events:
        rect = Rectangle(
            (extent[0], extent[2]),        # bottom-left corner (lon_min, lat_min)
            extent[1] - extent[0],         # width
            extent[3] - extent[2],         # height
            fill=False,                    # no fill, just outline
            edgecolor='black',
            linewidth=2.0,                 # thicker line for visibility
            zorder=5,                      # keep on top of base map
        )
        ax.add_patch(rect)

    # Determine name or fallback label
    event_name = cluster_event_map.get(cluster_id, f"Cluster {cluster_id}")

    # Filter points for this cluster
    cluster_points = clustered_claims[clustered_claims[optimal_cluster] == cluster_id]

    # GeoDataFrame for plotting
    gdf_claims_all = gpd.GeoDataFrame(
        cluster_points,
        geometry=gpd.points_from_xy(cluster_points['longitude'], cluster_points['latitude']),
        crs="EPSG:4326"
    )

    # shade critical states for this cluster (by GEOID)
    gdf_states.plot(ax=ax, facecolor='#F2F2F2', edgecolor='none', rasterized=True)
    fips_list = normalized_critical_states.get(cluster_id, [])
    if fips_list:
        # Ensure state GEOIDs are strings and 2-digit padded
        states_to_fill = gdf_states[gdf_states['GEOID'].astype(str).str.zfill(2).isin(fips_list)]
        if not states_to_fill.empty:
            states_to_fill.plot(ax=ax, facecolor=STATE_FILL, edgecolor='none', alpha=STATE_FILL_ALPHA, rasterized=True)

    # Plot boundaries and points (boundaries on top of fills, points on very top)
    gdf_states.boundary.plot(ax=ax, color='black', linewidth=0.5, rasterized=True)
    gdf_rivers.plot(ax=ax, color=RIVER_FILL, linewidth=0.2, rasterized=True)

    # colored if a critical event
    point_color = POINT_COLOR_CRITICAL if cluster_id in critical_events else POINT_COLOR_DEFAULT
    gdf_claims_all.plot(ax=ax, color=point_color, markersize=2, alpha=0.8, rasterized=True)

    # Format plot
    ax.set_xlim(extent[0], extent[1])
    ax.set_ylim(extent[2], extent[3])
    ax.set_title(event_name, fontsize=10)
    ax.axis("off")

    # Panel label
    ax.text(label_offset[0], label_offset[1], panel_labels[i], transform=ax.transAxes,
            ha='left', va='top', fontsize=12, fontweight='bold')

# Turn off any extra axes
for j in range(len(event_list), len(axs)):
    axs[j].axis("off")

if save:
    plt.savefig(f"Plots/Fig3_{test_case}.pdf", dpi=500, bbox_inches='tight')
plt.show()